# Parallel 3 IDK Cascades: Camera FPS

This notebook is based on **Parallel 3 IDK Cascades** and adds fixed-rate camera arrivals without changing the cascade, router, linked-list ordering, or work stealing.

## Imports

This cell imports the libraries used by the notebook. The worker import happens in the constants cell because it needs the project path first.

In [1]:
import json
import queue
import sys
import tarfile
import threading
import time
from collections import Counter, deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
from PIL import Image
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms

## Constants

Edit this cell to change models, devices, paths, worker counts, thresholds, sample counts, and camera rates. Device comments follow `MODEL_DEVICES`; the current configuration places all three models on MPS.

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RUNS_DIR = PROJECT_ROOT / "runs"
IMAGENETV2_DIR = PROJECT_ROOT / "ImageNet-V2 DataSet"
RUNS_DIR.mkdir(exist_ok=True)

VARIANT_ARCHIVES = {
    "matched-frequency": IMAGENETV2_DIR / "imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": IMAGENETV2_DIR / "imagenetv2-threshold0.7.tar.gz",
    "top-images": IMAGENETV2_DIR / "imagenetv2-top-images.tar.gz",
}

MODEL_A = "resnet18"
MODEL_B = "resnet34"
MODEL_C = "resnet152"

MODEL_A_DEVICE = "mps"
MODEL_B_DEVICE = "mps"
MODEL_C_DEVICE = "mps"

MODELS = (MODEL_A, MODEL_B, MODEL_C)
MPS_MODELS = (MODEL_B, MODEL_C)
MODEL_DEVICES = {
    MODEL_A: MODEL_A_DEVICE,
    MODEL_B: MODEL_B_DEVICE,
    MODEL_C: MODEL_C_DEVICE,
}

WORKER_LIMITS = {"cpu": 1, "mps": 3}
MAX_SAMPLES = 10000
MODEL_A_MAX_IN_FLIGHT = 1
INPUT_FPS_VALUES = 60
RUN_UNLIMITED_BASELINE = False
CAMERA_PREFETCH_SIZE = 64
RATE_TEST_MAX_SAMPLES = MAX_SAMPLES
SAVE_RATE_SWEEP_RESULTS = True
BATCH_SIZE = 1
CONFIDENCE_THRESHOLD = 0.9
MAX_IN_FLIGHT_PER_MPS_MODEL = 10

ROUTER_TRAIN_PREFIXES = ("matched", "top")
TEST_VARIANT = "threshold-0.7"
ROUTER_REQUIRES_CORRECT = False
SAVE_RESULTS = True
RESULTS_PATH = RUNS_DIR / "real_cpu_mps_worker_results.json"
PREDICTIONS_PATH = RUNS_DIR / "real_cpu_mps_worker_predictions.npz"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from real_time_mp_workers import model_worker

## Image Transform

This cell defines the ImageNet preprocessing used for router training and for the real-time test images.

In [3]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


## Dataset Loader

This cell reads ImageNetV2 rows from the local tar archives, applies the ImageNet transform, and returns a DataLoader for the real-time test.

In [4]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def label_from_key(key):
    return int(key.split("/")[1])


def stream_imagenet_v2_rows(variant, max_samples):
    emitted = 0
    with tarfile.open(VARIANT_ARCHIVES[variant], "r:*") as tar:
        for member in tar:
            if not member.isfile() or not member.name.lower().endswith(IMAGE_EXTENSIONS):
                continue

            image_file = tar.extractfile(member)
            if image_file is None:
                continue
            image = Image.open(image_file).convert("RGB")
            image_file.close()

            yield image, label_from_key(member.name)
            emitted += 1
            if emitted >= max_samples:
                break


class StreamingImageNetV2Dataset(IterableDataset):
    def __init__(self, variant, max_samples):
        self.variant = variant
        self.max_samples = int(max_samples)

    def __iter__(self):
        for image, label in stream_imagenet_v2_rows(self.variant, self.max_samples):
            yield preprocess(image), int(label)

    def __len__(self):
        return self.max_samples


def make_streaming_loader(variant, max_samples):
    dataset = StreamingImageNetV2Dataset(variant, max_samples)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=False,
    )
    return dataset, loader


## Probability Features

This cell converts model probabilities into the three router features: confidence, entropy, and margin between the top two probabilities.

In [5]:
def probability_features(probabilities):
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    top_two = np.partition(probabilities, -2, axis=1)[:, -2:]
    margin = top_two.max(axis=1) - top_two.min(axis=1)
    return np.column_stack([confidence, entropy, margin]).astype(np.float32)


def short_model_name(model_name):
    return model_name.replace("resnet", "RN").upper()


## Router Training Data

This cell builds router training data from the existing artifact NPZ files. It uses Model A probabilities as features and labels uncertain samples for Model B or Model C.

In [6]:
def load_cache(prefix, model_name):
    path = ARTIFACTS_DIR / f"{prefix}_{model_name}.npz"
    with np.load(path) as data:
        return {
            "probabilities": data["probabilities"],
            "predictions": data["predictions"],
            "labels": data["labels"],
        }


def cache_confidence(cache):
    return np.asarray(cache["probabilities"]).max(axis=1)


def build_router_training_data():
    feature_parts = []
    label_parts = []

    for prefix in ROUTER_TRAIN_PREFIXES:
        cache_a = load_cache(prefix, MODEL_A)
        cache_b = load_cache(prefix, MODEL_B)
        cache_c = load_cache(prefix, MODEL_C)

        features_a = probability_features(cache_a["probabilities"])
        model_a_uncertain = features_a[:, 0] < CONFIDENCE_THRESHOLD
        model_b_ok = cache_confidence(cache_b) >= CONFIDENCE_THRESHOLD
        model_c_ok = cache_confidence(cache_c) >= CONFIDENCE_THRESHOLD

        if ROUTER_REQUIRES_CORRECT:
            model_b_ok = model_b_ok & (cache_b["predictions"] == cache_a["labels"])
            model_c_ok = model_c_ok & (cache_c["predictions"] == cache_a["labels"])

        keep = model_a_uncertain & (model_b_ok | model_c_ok)
        route_labels = np.where(model_b_ok[keep], 0, 1).astype(np.int64)

        feature_parts.append(features_a[keep])
        label_parts.append(route_labels)

    router_features = np.concatenate(feature_parts)
    router_labels = np.concatenate(label_parts)
    if len(router_labels) == 0:
        raise ValueError("No router training rows found")
    return router_features, router_labels


router_train_features, router_train_labels = build_router_training_data()
print("Router training samples:", len(router_train_labels))
print(f"{MODEL_B} labels:", int(np.count_nonzero(router_train_labels == 0)))
print(f"{MODEL_C} labels:", int(np.count_nonzero(router_train_labels == 1)))


Router training samples: 2589
resnet34 labels: 2311
resnet152 labels: 278


## Random Forest Router

Run this cell to use a Random Forest router. To switch routers during Run All, comment out this cell and uncomment one of the next router cells.

In [7]:
# router_name = "rf"
# router = RandomForestClassifier(
#     n_estimators=100,
#     max_depth=6,
#     min_samples_leaf=20,
#     class_weight="balanced",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


## Extra Trees Router

Uncomment and run this cell to use an Extra Trees router. Comment out the other router cells first.

In [8]:
# router_name = "extratree"
# router = ExtraTreesClassifier(
#     n_estimators=100,
#     max_depth=6,
#     min_samples_leaf=20,
#     class_weight="balanced",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


## XGBoost Router

Uncomment and run this cell to use an XGBoost router. Comment out the other router cells first. Install xgboost before using it.

In [9]:
# Install xgboost before uncommenting this cell.
# from xgboost import XGBClassifier
#
# router_name = "xgboost"
# router = XGBClassifier(
#     n_estimators=100,
#     max_depth=4,
#     learning_rate=0.1,
#     eval_metric="logloss",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


## SVM Router

Uncomment and run this cell to use an SVM router. Comment out the other router cells first.

In [10]:
router_name = "svm"
router = SVC(kernel="rbf", C=2.0, gamma="scale", class_weight="balanced")
router.fit(router_train_features, router_train_labels)
print("Router:", router_name)


Router: svm


## Test Dataset

This cell checks the configured devices and reports the dataset. Each experiment creates its own fresh streaming loader so every rate sees the same image order.

In [11]:
if any(device == "mps" for device in MODEL_DEVICES.values()) and not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required by MODEL_DEVICES")

real_test_dataset, _ = make_streaming_loader(TEST_VARIANT, MAX_SAMPLES)
real_sample_count = len(real_test_dataset)

print("Local samples:", real_sample_count)
print("Dataset:", VARIANT_ARCHIVES[TEST_VARIANT])
print("Worker limits:", WORKER_LIMITS)
print("Worker model devices:", MODEL_DEVICES)

Local samples: 10000
Dataset: /Users/abhinavgupta/dynamic-idk-cascades/ImageNet-V2 DataSet/imagenetv2-threshold0.7.tar.gz
Worker limits: {'cpu': 1, 'mps': 3}
Worker model devices: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet152': 'mps'}


## Doubly Linked List

This cell defines the heavy-model waiting queue. Model B jobs stay at the head side; Model C jobs stay at the tail side; Model C can steal a Model B job when no Model C job is waiting.

In [12]:
class JobNode:
    def __init__(self, sample_index, images, assigned_model):
        self.sample_index = sample_index
        self.images = images
        self.assigned_model = assigned_model
        self.prev = None
        self.next = None


class DoublyLinkedList:
    def __init__(self, model_b, model_c):
        self.model_b = model_b
        self.model_c = model_c
        self.head = None
        self.tail = None
        self.last_model_b = None
        self.size = 0

    def _insert_empty(self, node):
        self.head = node
        self.tail = node
        self.size = 1
        if node.assigned_model == self.model_b:
            self.last_model_b = node

    def _insert_before(self, anchor, node):
        node.prev = anchor.prev
        node.next = anchor
        if anchor.prev is None:
            self.head = node
        else:
            anchor.prev.next = node
        anchor.prev = node
        self.size += 1

    def _insert_after(self, anchor, node):
        node.prev = anchor
        node.next = anchor.next
        if anchor.next is None:
            self.tail = node
        else:
            anchor.next.prev = node
        anchor.next = node
        self.size += 1

    def _remove(self, node):
        if node.prev is None:
            self.head = node.next
        else:
            node.prev.next = node.next

        if node.next is None:
            self.tail = node.prev
        else:
            node.next.prev = node.prev

        if node is self.last_model_b:
            self.last_model_b = node.prev if node.prev and node.prev.assigned_model == self.model_b else None

        node.prev = None
        node.next = None
        self.size -= 1

        if self.size == 0:
            self.head = None
            self.tail = None
            self.last_model_b = None

        return node

    def insert_middle(self, sample_index, images, assigned_model):
        node = JobNode(sample_index, images, assigned_model)
        if self.size == 0:
            self._insert_empty(node)
            return node

        if assigned_model == self.model_b:
            if self.last_model_b is None:
                self._insert_before(self.head, node)
            else:
                self._insert_after(self.last_model_b, node)
            self.last_model_b = node
        elif assigned_model == self.model_c:
            if self.last_model_b is None:
                self._insert_before(self.head, node)
            else:
                self._insert_after(self.last_model_b, node)
        else:
            raise ValueError(f"Unknown heavy model: {assigned_model}")

        return node

    def pop_for_model_b(self):
        if self.head is None or self.head.assigned_model != self.model_b:
            return None
        return self._remove(self.head)

    def pop_for_model_c_or_steal_model_b(self):
        if self.tail is not None and self.tail.assigned_model == self.model_c:
            return self._remove(self.tail), False

        node = self.pop_for_model_b()
        if node is None:
            return None, False
        return node, True


## Router Function

This cell converts Model A probabilities into router features and returns Model B or Model C.

In [13]:
def route_from_model_a(probabilities):
    route_label = int(router.predict(probability_features(probabilities[None, :]))[0])
    return MODEL_B if route_label == 0 else MODEL_C


## Camera-Aware Real-Time Run

The model workers, confidence threshold, router, doubly linked list, and Model C work stealing are unchanged. A bounded producer thread decodes frames ahead of time, while the scheduler admits fixed-rate frames from an absolute `perf_counter()` clock.

In [14]:
PREFETCH_END = object()


def _prefetch_frames(loader, output_queue, stop_event):
    try:
        for sample_index, (images, batch_labels) in enumerate(loader):
            item = (sample_index, images, int(batch_labels.item()))
            while not stop_event.is_set():
                try:
                    output_queue.put(item, timeout=0.1)
                    break
                except queue.Full:
                    pass
            if stop_event.is_set():
                return
        item = PREFETCH_END
    except Exception as exc:
        item = ("producer_error", repr(exc))

    while not stop_event.is_set():
        try:
            output_queue.put(item, timeout=0.1)
            return
        except queue.Full:
            pass


def _percentile(values, percentile):
    return float(np.percentile(values, percentile))


def run_real_time_test(input_fps, max_samples):
    if input_fps is not None and input_fps <= 0:
        raise ValueError("input_fps must be positive or None")
    if max_samples <= 0:
        raise ValueError("max_samples must be positive")

    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    job_queues = {model_name: ctx.Queue() for model_name in MODELS}
    result_queue = ctx.Queue()
    processes = [
        ctx.Process(
            target=model_worker,
            args=(model_name, MODEL_DEVICES[model_name], job_queues[model_name], result_queue, True),
        )
        for model_name in MODELS
    ]

    _, loader = make_streaming_loader(TEST_VARIANT, max_samples)
    prepared_frames = queue.Queue(maxsize=CAMERA_PREFETCH_SIZE)
    producer_stop = threading.Event()
    producer = threading.Thread(
        target=_prefetch_frames,
        args=(loader, prepared_frames, producer_stop),
        name="camera-prefetch",
        daemon=True,
    )

    for process in processes:
        process.start()
    producer.start()

    try:
        ready_models = set()
        while ready_models != set(MODELS):
            message = result_queue.get()
            if message[0] == "error":
                _, model_name, error = message
                raise RuntimeError(f"{model_name} worker failed: {error}")
            if message[0] != "ready" or message[1] not in MODELS:
                raise RuntimeError(f"Unexpected worker startup message: {message[:2]}")
            ready_models.add(message[1])

        total_samples = int(max_samples)
        labels = np.full(total_samples, -1, dtype=np.int64)
        final_predictions = np.full(total_samples, -1, dtype=np.int64)
        chosen_models = np.full(total_samples, "", dtype="<U32")
        scheduled_arrivals = np.full(total_samples, np.nan)
        actual_arrivals = np.full(total_samples, np.nan)
        model_a_dispatches = np.full(total_samples, np.nan)
        completion_times = np.full(total_samples, np.nan)

        execution_count_by_model = Counter()
        execution_time_ms_by_model = Counter()
        heavy_route_count_by_model = Counter()
        stolen_job_count_by_model = Counter()
        in_flight_by_model = Counter()

        sample_states = {}
        seen_sample_indices = set()
        camera_input_queue = deque()
        doubly_linked_list = DoublyLinkedList(MODEL_B, MODEL_C)
        backlog_timeline = []
        next_arrival_index = 0
        completed_sample_count = 0
        actual_arrival_count = 0
        heavy_queue_max_size = 0
        maximum_camera_queue_size = 0
        maximum_unfinished_frames = 0
        router_call_count = 0
        router_time_ms_total = 0.0
        source_starvation_count = 0
        starvation_started = set()
        source_starvation_ms = []
        unfinished_at_last_arrival = None
        completed_at_last_arrival = None

        frame_period = None if input_fps is None else 1.0 / input_fps
        capture_start = time.perf_counter()
        run_start = capture_start
        last_scheduled_arrival = (
            None if input_fps is None else capture_start + (total_samples - 1) * frame_period
        )

        def scheduled_count(now):
            if input_fps is None:
                return actual_arrival_count
            if now < capture_start:
                return 0
            return min(total_samples, int((now - capture_start) / frame_period) + 1)

        def unfinished_count(now):
            return scheduled_count(now) - completed_sample_count

        def record_timeline(now):
            nonlocal maximum_unfinished_frames
            elapsed = now - capture_start
            unfinished = unfinished_count(now)
            maximum_unfinished_frames = max(maximum_unfinished_frames, unfinished)
            if input_fps is None or now <= last_scheduled_arrival:
                backlog_timeline.append(
                    (
                        elapsed,
                        unfinished,
                        len(camera_input_queue),
                        doubly_linked_list.size,
                        completed_sample_count,
                        scheduled_count(now),
                    )
                )

        def get_prepared(expected_index):
            try:
                item = prepared_frames.get_nowait()
            except queue.Empty:
                return None
            if item is PREFETCH_END:
                raise RuntimeError(f"Dataset ended after {expected_index} of {total_samples} samples")
            if item[0] == "producer_error":
                raise RuntimeError(f"Camera producer failed: {item[1]}")
            if item[0] != expected_index:
                raise RuntimeError(f"Expected sample {expected_index}, got {item[0]}")
            return item

        def enqueue_arrivals(now):
            nonlocal next_arrival_index, actual_arrival_count
            nonlocal source_starvation_count, maximum_camera_queue_size

            if input_fps is None:
                due = min(total_samples, next_arrival_index + int(in_flight_by_model[MODEL_A] < MODEL_A_MAX_IN_FLIGHT))
            else:
                due = scheduled_count(now)

            while next_arrival_index < due:
                item = get_prepared(next_arrival_index)
                scheduled = now if input_fps is None else capture_start + next_arrival_index * frame_period
                if item is None:
                    if next_arrival_index not in starvation_started:
                        starvation_started.add(next_arrival_index)
                        source_starvation_count += 1
                    break

                sample_index, images, label = item
                if sample_index in seen_sample_indices:
                    raise AssertionError(f"Duplicate sample index {sample_index}")
                seen_sample_indices.add(sample_index)
                actual = time.perf_counter()
                if sample_index in starvation_started:
                    source_starvation_ms.append((actual - scheduled) * 1000.0)
                labels[sample_index] = label
                scheduled_arrivals[sample_index] = scheduled
                actual_arrivals[sample_index] = actual
                camera_input_queue.append((sample_index, images, label, scheduled, actual))
                next_arrival_index += 1
                actual_arrival_count += 1
                maximum_camera_queue_size = max(maximum_camera_queue_size, len(camera_input_queue))

        def dispatch_model_a():
            while camera_input_queue and in_flight_by_model[MODEL_A] < MODEL_A_MAX_IN_FLIGHT:
                sample_index, images, label, scheduled, actual = camera_input_queue.popleft()
                dispatch_time = time.perf_counter()
                if dispatch_time < scheduled:
                    raise AssertionError("Frame dispatched before scheduled arrival")
                sample_states[sample_index] = {"images": images}
                model_a_dispatches[sample_index] = dispatch_time
                job_queues[MODEL_A].put((sample_index, images))
                in_flight_by_model[MODEL_A] += 1
                execution_count_by_model[MODEL_A] += 1

        def heavy_backlog_size():
            return doubly_linked_list.size + sum(in_flight_by_model[name] for name in MPS_MODELS)

        def dispatch_mps_jobs(max_size):
            for model_name in MPS_MODELS:
                while in_flight_by_model[model_name] < MAX_IN_FLIGHT_PER_MPS_MODEL:
                    if model_name == MODEL_B:
                        node = doubly_linked_list.pop_for_model_b()
                        stole_model_b_job = False
                    else:
                        node, stole_model_b_job = doubly_linked_list.pop_for_model_c_or_steal_model_b()
                    if node is None:
                        break
                    job_queues[model_name].put((node.sample_index, node.images))
                    in_flight_by_model[model_name] += 1
                    execution_count_by_model[model_name] += 1
                    if stole_model_b_job:
                        stolen_job_count_by_model[MODEL_C] += 1
                    max_size = max(max_size, heavy_backlog_size())
            return max_size

        def process_result(message):
            nonlocal completed_sample_count, router_call_count, router_time_ms_total
            nonlocal heavy_queue_max_size

            if message[0] == "error":
                _, model_name, error = message
                raise RuntimeError(f"{model_name} worker failed: {error}")
            if message[0] == "ready":
                raise RuntimeError(f"Duplicate ready message from {message[1]}")

            sample_index, model_name, probabilities, prediction, confidence, elapsed_ms = message
            if sample_index not in sample_states or model_name not in MODELS:
                raise RuntimeError(f"Invalid worker result for sample {sample_index}")
            in_flight_by_model[model_name] -= 1
            if in_flight_by_model[model_name] < 0:
                raise AssertionError(f"Negative in-flight count for {model_name}")
            execution_time_ms_by_model[model_name] += elapsed_ms

            if model_name == MODEL_A and confidence < CONFIDENCE_THRESHOLD:
                router_start = time.perf_counter()
                assigned_model = route_from_model_a(probabilities)
                router_time_ms_total += (time.perf_counter() - router_start) * 1000.0
                router_call_count += 1
                heavy_route_count_by_model[assigned_model] += 1
                doubly_linked_list.insert_middle(
                    sample_index, sample_states[sample_index]["images"], assigned_model
                )
                sample_states[sample_index]["images"] = None
                heavy_queue_max_size = max(heavy_queue_max_size, heavy_backlog_size())
            else:
                if final_predictions[sample_index] != -1:
                    raise AssertionError(f"Duplicate final prediction for sample {sample_index}")
                final_predictions[sample_index] = prediction
                chosen_models[sample_index] = model_name
                completion_times[sample_index] = time.perf_counter()
                sample_states.pop(sample_index)
                completed_sample_count += 1

            heavy_queue_max_size = dispatch_mps_jobs(heavy_queue_max_size)

        while completed_sample_count < total_samples:
            now = time.perf_counter()
            enqueue_arrivals(now)
            dispatch_model_a()

            while True:
                try:
                    process_result(result_queue.get_nowait())
                    dispatch_model_a()
                except queue.Empty:
                    break

            now = time.perf_counter()
            if input_fps is not None and unfinished_at_last_arrival is None and now >= last_scheduled_arrival:
                unfinished_at_last_arrival = total_samples - completed_sample_count
                completed_at_last_arrival = completed_sample_count
            record_timeline(now)

            if completed_sample_count == total_samples:
                break

            if input_fps is not None and next_arrival_index < total_samples:
                next_time = capture_start + next_arrival_index * frame_period
                timeout = max(0.001, next_time - time.perf_counter())
                if next_time <= time.perf_counter():
                    timeout = 0.01
            elif input_fps is None and next_arrival_index < total_samples and not camera_input_queue:
                timeout = 0.01
            else:
                timeout = None

            try:
                message = result_queue.get() if timeout is None else result_queue.get(timeout=timeout)
                process_result(message)
            except queue.Empty:
                pass

        final_completion = time.perf_counter()
        total_wall_time_seconds = final_completion - run_start
        if unfinished_at_last_arrival is None:
            unfinished_at_last_arrival = total_samples - completed_sample_count
            completed_at_last_arrival = completed_sample_count

        source_delay_ms = (actual_arrivals - scheduled_arrivals) * 1000.0
        model_a_queue_wait_ms = (model_a_dispatches - scheduled_arrivals) * 1000.0
        end_to_end_latency_ms = (completion_times - scheduled_arrivals) * 1000.0
        actual_capture_duration = max(0.0, actual_arrivals[-1] - actual_arrivals[0])
        realized_input_fps = (
            (total_samples - 1) / actual_capture_duration
            if total_samples > 1 and actual_capture_duration > 0
            else 0.0
        )
        intended_capture_duration = (
            0.0 if input_fps is None else (total_samples - 1) * frame_period
        )
        drain_time = (
            0.0 if input_fps is None else max(0.0, final_completion - last_scheduled_arrival)
        )

        timeline = np.asarray(backlog_timeline, dtype=np.float64)
        backlog_growth = None
        keeps_up = None
        backlog_threshold = None
        if input_fps is not None:
            warmup_end = intended_capture_duration * 0.10
            trend_rows = timeline[timeline[:, 0] >= warmup_end]
            if len(trend_rows) >= 2 and np.ptp(trend_rows[:, 0]) > 0:
                backlog_growth = float(np.polyfit(trend_rows[:, 0], trend_rows[:, 1], 1)[0])
            else:
                backlog_growth = 0.0
            backlog_threshold = max(0.5, input_fps * 0.02)
            rate_error = abs(realized_input_fps - input_fps) / input_fps
            keeps_up = bool(backlog_growth <= backlog_threshold and rate_error <= 0.05)

        assert completed_sample_count == total_samples
        assert seen_sample_indices == set(range(total_samples))
        assert np.all(final_predictions >= 0)
        assert np.all(labels >= 0)
        assert np.all(chosen_models != "")
        assert not np.isnan(scheduled_arrivals).any()
        assert not np.isnan(actual_arrivals).any()
        assert not np.isnan(model_a_dispatches).any()
        assert not np.isnan(completion_times).any()
        assert np.all(model_a_dispatches >= scheduled_arrivals)
        assert all(in_flight_by_model[name] == 0 for name in MODELS)
        assert not camera_input_queue
        assert doubly_linked_list.size == 0
        assert not sample_states
        assert len(source_starvation_ms) == source_starvation_count

        correct_predictions = int(np.count_nonzero(final_predictions == labels))
        total_execution_time_ms = {
            name: float(execution_time_ms_by_model[name]) for name in MODELS
        }
        mean_execution_time_ms = {
            name: (
                float(execution_time_ms_by_model[name] / execution_count_by_model[name])
                if execution_count_by_model[name]
                else 0.0
            )
            for name in MODELS
        }
        results = {
            "input_mode": "unlimited" if input_fps is None else f"{input_fps:g}fps",
            "configured_input_fps": None if input_fps is None else float(input_fps),
            "frame_period_ms": None if input_fps is None else float(frame_period * 1000.0),
            "total_samples": total_samples,
            "intended_capture_duration_seconds": float(intended_capture_duration),
            "realized_capture_duration_seconds": float(actual_capture_duration),
            "realized_input_fps": float(realized_input_fps),
            "total_wall_time_seconds": float(total_wall_time_seconds),
            "drain_time_after_last_arrival_seconds": float(drain_time),
            "completed_throughput_fps": float(total_samples / total_wall_time_seconds),
            "throughput_fps": float(total_samples / total_wall_time_seconds),
            "active_capture_completion_fps": (
                None
                if input_fps is None or intended_capture_duration == 0
                else float(completed_at_last_arrival / intended_capture_duration)
            ),
            "source_starvation_count": int(source_starvation_count),
            "total_source_starvation_time_ms": float(sum(source_starvation_ms)),
            "maximum_source_starvation_time_ms": float(max(source_starvation_ms, default=0.0)),
            "mean_end_to_end_latency_ms": float(end_to_end_latency_ms.mean()),
            "p50_end_to_end_latency_ms": _percentile(end_to_end_latency_ms, 50),
            "p95_end_to_end_latency_ms": _percentile(end_to_end_latency_ms, 95),
            "p99_end_to_end_latency_ms": _percentile(end_to_end_latency_ms, 99),
            "maximum_end_to_end_latency_ms": float(end_to_end_latency_ms.max()),
            "mean_latency_ms": float(end_to_end_latency_ms.mean()),
            "mean_model_a_queue_wait_ms": float(model_a_queue_wait_ms.mean()),
            "p95_model_a_queue_wait_ms": _percentile(model_a_queue_wait_ms, 95),
            "maximum_model_a_queue_wait_ms": float(model_a_queue_wait_ms.max()),
            "maximum_camera_input_queue_size": int(maximum_camera_queue_size),
            "maximum_total_unfinished_frames": int(maximum_unfinished_frames),
            "unfinished_frames_at_last_arrival": int(unfinished_at_last_arrival),
            "unfinished_backlog_growth_fps": backlog_growth,
            "backlog_growth_threshold_fps": backlog_threshold,
            "keeps_up_with_input": keeps_up,
            "keeps_up_definition": (
                "After 10% warm-up, backlog slope <= max(0.5, configured_input_fps * 0.02) "
                "and realized input FPS is within 5% of target."
            ),
            "accuracy": float(correct_predictions / total_samples),
            "correct_predictions": correct_predictions,
            "model_a_max_in_flight": MODEL_A_MAX_IN_FLIGHT,
            "worker_limits": dict(WORKER_LIMITS),
            "device_by_model": dict(MODEL_DEVICES),
            "router_name": router_name,
            "router_call_count": int(router_call_count),
            "total_router_time_ms": float(router_time_ms_total),
            "mean_router_time_ms": float(router_time_ms_total / router_call_count) if router_call_count else 0.0,
            "stolen_job_count_by_model": {name: int(stolen_job_count_by_model[name]) for name in MODELS},
            "final_prediction_count_by_model": {name: int(np.count_nonzero(chosen_models == name)) for name in MODELS},
            "execution_count_by_model": {name: int(execution_count_by_model[name]) for name in MODELS},
            "total_execution_time_ms_by_model": total_execution_time_ms,
            "mean_execution_time_ms_by_model": mean_execution_time_ms,
            "idle_time_ms_by_model": {
                name: max(0.0, total_wall_time_seconds * 1000.0 - total_execution_time_ms[name])
                for name in MODELS
            },
            "heavy_route_count_by_model": {name: int(heavy_route_count_by_model[name]) for name in MPS_MODELS},
            "heavy_queue_max_size": int(heavy_queue_max_size),
        }
        timing_data = {
            "labels": labels,
            "scheduled_arrival_times_seconds": scheduled_arrivals - capture_start,
            "actual_arrival_times_seconds": actual_arrivals - capture_start,
            "model_a_dispatch_times_seconds": model_a_dispatches - capture_start,
            "completion_times_seconds": completion_times - capture_start,
            "source_delay_ms": source_delay_ms,
            "model_a_queue_wait_ms": model_a_queue_wait_ms,
            "end_to_end_latency_ms": end_to_end_latency_ms,
        }
        backlog_data = {
            "backlog_timeline_seconds": timeline[:, 0],
            "backlog_timeline_frame_counts": timeline[:, 1].astype(np.int64),
            "camera_queue_timeline_frame_counts": timeline[:, 2].astype(np.int64),
            "heavy_queue_timeline_frame_counts": timeline[:, 3].astype(np.int64),
            "completed_timeline_frame_counts": timeline[:, 4].astype(np.int64),
            "arrived_timeline_frame_counts": timeline[:, 5].astype(np.int64),
        }
        return results, final_predictions, chosen_models, timing_data, backlog_data

    finally:
        producer_stop.set()
        while True:
            try:
                prepared_frames.get_nowait()
            except queue.Empty:
                break
        producer.join(timeout=5)
        for job_queue in job_queues.values():
            job_queue.put(None)
        for process in processes:
            process.join(timeout=10)
            if process.is_alive():
                process.terminate()
                process.join()
        assert not producer.is_alive(), "Producer thread did not stop"

## Offline Baseline and Fixed-Rate Camera Sweep

The previous experiment was an always-available offline workload: another image was submitted whenever Model A had capacity. The fixed-rate runs instead schedule camera frames from one absolute clock. This exposes growing queues when inference cannot keep up.

All frames are processed. This experiment measures overload; it does not apply frame skipping. Completion throughput, end-to-end latency, backlog growth, and drain time answer different questions, so the sweep reports all four.

In [15]:
def save_experiment(results, predictions, chosen_models, timing_data, backlog_data):
    suffix = "unlimited" if results["configured_input_fps"] is None else f'{results["configured_input_fps"]:g}fps'
    results_path = RUNS_DIR / f"real_cpu_mps_{suffix}_results.json"
    predictions_path = RUNS_DIR / f"real_cpu_mps_{suffix}_predictions.npz"
    results_path.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
    np.savez_compressed(
        predictions_path,
        predictions=predictions,
        chosen_models=chosen_models,
        **timing_data,
        **backlog_data,
    )


def print_experiment_summary(results):
    print("Camera-FPS multiprocessing runtime test")
    print("Worker model devices:", results["device_by_model"])
    print("Total samples:", results["total_samples"])
    print("Accuracy:", round(results["accuracy"], 4))
    print("Mean latency (ms):", round(results["mean_end_to_end_latency_ms"], 3))
    print("P95 latency (ms):", round(results["p95_end_to_end_latency_ms"], 3))
    print("Configured camera FPS cap:", results["configured_input_fps"])
    print("Realized camera FPS:", round(results["realized_input_fps"], 3))
    print("Throughput (FPS):", round(results["completed_throughput_fps"], 3))
    print("Keeps up with input:", results["keeps_up_with_input"])
    print("Max camera queue:", results["maximum_camera_input_queue_size"])
    print("Backlog at last arrival:", results["unfinished_frames_at_last_arrival"])
    print("Drain time (seconds):", round(results["drain_time_after_last_arrival_seconds"], 3))
    print()
    print("Final prediction count by model:")
    for model_name in MODELS:
        print(f'  {model_name}: {results["final_prediction_count_by_model"][model_name]}')
    print("Execution count by model:")
    for model_name in MODELS:
        print(f'  {model_name}: {results["execution_count_by_model"][model_name]}')
    print("Mean execution time by model (ms):")
    for model_name in MODELS:
        value = results["mean_execution_time_ms_by_model"][model_name]
        print(f"  {model_name}: {value:.3f}")
    print()


experiment_results = []
experiment_outputs = {}
expected_labels = None

input_fps_values = (INPUT_FPS_VALUES,) if isinstance(INPUT_FPS_VALUES, (int, float)) else tuple(INPUT_FPS_VALUES)
rates = ([None] if RUN_UNLIMITED_BASELINE else []) + list(input_fps_values)
for input_fps in rates:
    output = run_real_time_test(input_fps, RATE_TEST_MAX_SAMPLES)
    results, predictions, chosen_models, timing_data, backlog_data = output
    labels = timing_data["labels"]
    if expected_labels is None:
        expected_labels = labels.copy()
    else:
        assert len(labels) == len(expected_labels)
        assert np.array_equal(labels, expected_labels), "Label order changed between rates"

    experiment_results.append(results)
    experiment_outputs[results["input_mode"]] = output
    print_experiment_summary(results)
    if SAVE_RATE_SWEEP_RESULTS:
        save_experiment(*output)

if SAVE_RATE_SWEEP_RESULTS:
    (RUNS_DIR / "real_cpu_mps_rate_sweep.json").write_text(
        json.dumps(experiment_results, indent=2) + "\n", encoding="utf-8"
    )
    pd.DataFrame(experiment_results).to_csv(
        RUNS_DIR / "real_cpu_mps_rate_sweep.csv", index=False
    )

Camera-FPS multiprocessing runtime test
Worker model devices: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet152': 'mps'}
Total samples: 10000
Accuracy: 0.7642
Mean latency (ms): 19131.787
P95 latency (ms): 63104.729
Configured camera FPS cap: 60.0
Realized camera FPS: 60.0
Throughput (FPS): 43.11
Keeps up with input: False
Max camera queue: 1023
Backlog at last arrival: 1776
Drain time (seconds): 65.316

Final prediction count by model:
  resnet18: 3722
  resnet34: 3050
  resnet152: 3228
Execution count by model:
  resnet18: 10000
  resnet34: 3050
  resnet152: 3228
Mean execution time by model (ms):
  resnet18: 17.339
  resnet34: 19.961
  resnet152: 71.333



## Research Summary

One row per experiment summarizes arrival fidelity, completion rate, accuracy, latency, queue growth, drain time, starvation, and the reproducible keeps-up decision.

The runtime report is printed directly by the sweep cell above, with one measurement per line. CSV and JSON files are still saved when SAVE_RATE_SWEEP_RESULTS is enabled.

## Rate-Sweep Plots

Unlimited input is omitted from plots whose horizontal axis is a numeric camera rate.

## Backlog Timelines

Each fixed-rate plot separates all unfinished frames from frames waiting before Model A and jobs waiting in the heavy-model linked list.

In [16]:
for result in fixed_results:
    output = experiment_outputs[result["input_mode"]]
    backlog = output[4]
    seconds = backlog["backlog_timeline_seconds"]
    plt.figure(figsize=(8, 4))
    plt.plot(seconds, backlog["backlog_timeline_frame_counts"], label="Unfinished frames")
    plt.plot(seconds, backlog["camera_queue_timeline_frame_counts"], label="Waiting before Model A")
    plt.plot(seconds, backlog["heavy_queue_timeline_frame_counts"], label="Heavy-model queue")
    plt.xlabel("Elapsed capture time (sec)")
    plt.ylabel("Frames")
    plt.title(f'{result["configured_input_fps"]:g} FPS Backlog')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

NameError: name 'fixed_results' is not defined